In [1]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate

In [9]:
# def get_weather(lon, lat):
#     print("call an api ... ")
    
function = {
    "name": "get_weather",
    "description": "function that takes longitude and latitude to find the weather of a place",
    "parameters": {
        "type": "object",
        "properties": {
            "lon": {
                "type": "string",
                "description": "The longitude coordinate"},
            "lat": {
                "type": "string",
                "description": "The latitude coordinate"},
            },
        },
    "required": ["lon", "lat"],
}


In [20]:
function = {
    "name": "create_quiz",
    "description": "function that takes a list of questions and answers and returns a quiz",
    "parameters": {
        "type": "object",
        "properties": {
            "questions": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "question": {
                            "type": "string",
                        },
                        "answers": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "answer": {
                                        "type": "string",
                                    },
                                    "correct": {
                                        "type": "boolean",
                                    },
                                },
                                "required": ["answer", "correct"],
                            },
                        },
                    },
                    "required": ["question", "answers"],
                },
            }
        },
        "required": ["questions"],
    },
}


In [3]:
import os
import openai
os.environ["OPENAI_API_KEY"] = "" #모두의 연구소에서 발급
openai.api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
notebook.ipynb

In [21]:
llm = ChatOpenAI(
    temperature= 0.1,
).bind(
    #function_call={"name": "get_weather"}, # 모델이 강제로 함수를 사용하도록
    #function_call="auto", # 모델이 필요에 따라 함소호출여부를 선택하여 사용하도록
    function_call={
        "name": "create_quiz",
    },
    functions=[
        function
    ]
)

In [22]:
prompt = PromptTemplate.from_template("Make a quiz about {city}")

In [23]:
chain = prompt | llm

In [24]:
response = chain.invoke({
    "city":"rome"
})

response = response.additional_kwargs["function_call"]["arguments"]

In [25]:
response

'{"questions":[{"question":"What year was Rome founded?","answers":[{"answer":"753 BC","correct":true},{"answer":"476 AD","correct":false},{"answer":"1492 AD","correct":false}]},{"question":"Who was the first emperor of Rome?","answers":[{"answer":"Julius Caesar","correct":false},{"answer":"Augustus","correct":true},{"answer":"Nero","correct":false}]},{"question":"What famous structure in Rome was built by the ancient Romans?","answers":[{"answer":"Eiffel Tower","correct":false},{"answer":"Colosseum","correct":true},{"answer":"Big Ben","correct":false}]}]}'

In [26]:
import json

r = json.loads(response)

In [27]:
r

{'questions': [{'question': 'What year was Rome founded?',
   'answers': [{'answer': '753 BC', 'correct': True},
    {'answer': '476 AD', 'correct': False},
    {'answer': '1492 AD', 'correct': False}]},
  {'question': 'Who was the first emperor of Rome?',
   'answers': [{'answer': 'Julius Caesar', 'correct': False},
    {'answer': 'Augustus', 'correct': True},
    {'answer': 'Nero', 'correct': False}]},
  {'question': 'What famous structure in Rome was built by the ancient Romans?',
   'answers': [{'answer': 'Eiffel Tower', 'correct': False},
    {'answer': 'Colosseum', 'correct': True},
    {'answer': 'Big Ben', 'correct': False}]}]}

In [31]:
for question in json.loads(response)["questions"]:
    print(question)

{'question': 'What year was Rome founded?', 'answers': [{'answer': '753 BC', 'correct': True}, {'answer': '476 AD', 'correct': False}, {'answer': '1492 AD', 'correct': False}]}
{'question': 'Who was the first emperor of Rome?', 'answers': [{'answer': 'Julius Caesar', 'correct': False}, {'answer': 'Augustus', 'correct': True}, {'answer': 'Nero', 'correct': False}]}
{'question': 'What famous structure in Rome was built by the ancient Romans?', 'answers': [{'answer': 'Eiffel Tower', 'correct': False}, {'answer': 'Colosseum', 'correct': True}, {'answer': 'Big Ben', 'correct': False}]}


In [ ]:
import json
import subprocess
from pydub import AudioSegment #오디오 파일을 load


def extract_audio_from_video(video_path, audio_path):
    command = [
        "ffmpeg\", "-i", video_path, "-vn", audio_path,
    ]
    
    subprocess.run(command)
    
extract_audio_from_video(
    "./files/podcast.mp4",
    "./files/podcast.mp3",
)


track = AudioSegment.from_mp3("./files/podcast.mp3\")

#track.duraion_seconds #트랙의 총 실행시간 확인

ten_minutes = 10 * 60 * 1000 # pydub은 밀리초 단위이므로 1000을 곱한다.


In [ ]:

##11.2

import math\n",
    "\n",
    "chunks = math.ceil(len(track) / ten_minutes)\n",
    "\n",
    "for i in range(chunks):\n",
    "    start_time = i * ten_minutes\n",
    "    end_time = (i + 1) * ten_minutes\n",
    "\n",
    "    chunk = track[start_time:end_time]\n",
    "\n",
    "    chunk.export(f\"./files/chunks/chunk_{i}.mp3\", format=\"mp3\")"
   ]

In [ ]:
  #11.3 
  
def cut_audio_in_chunks(audio_path, chunk_size, chunks_folder): #11.2의 코드를 한 함수에 넣음
  track = AudioSegment.from_mp3(audio_path)\n",
  chunk_len = chunk_size * 60 * 1000\n",
  chunks = math.ceil(len(track) / chunk_len)\n",
  for i in range(chunks):\n",
    "        start_time = i * chunk_len\n",
    "        end_time = (i + 1) * chunk_len\n",
    "\n",
    "        chunk = track[start_time:end_time]\n",

In [ ]:
from typing import final
import openai

import glob #패턴을 사용하여 디렉토리 내부의 파일을 검색하도록 함
    "\n",
    "    chunk = track[start_time:end_time]\n",
    "\n",
    "    chunk.export(f\"./files/chunks/chunk_{i}.mp3\", format=\"mp3\")"
    "def transcribe_chunks(chunk_folder, destination):\n",
    "    files = glob.glob(f\"{chunk_folder}/*.mp3\")\n",
    "    final_transcript = \"\"\n",
    "    for file in files:\n",
    "        with open(file, \"rb\") as audio_file:\n",
    "            transcript = openai.Audio.transcribe(\n",
    "                \"whisper-1\",\n",
    "                audio_file,\n",
    "            )\n",
    "            final_transcript += transcript[\"text\"]\n",
    "    with open(destination, \"w\") as file:\n",
    "        file.write(final_transcript)"
   ]